# **Text Classification with Generative Models (FLAN-T5) & Multi-Dimensional Audience/Geography Framing**
### *Completed Solution & Advanced Extension by Gwendalynn Lim (婉婷) — Singapore Institute of Technology (SIT) · Nanyang Technological University (NTU)*


**Learning Objectives**

By the end of this exercise, you will be able to:
- Load and use a pre-trained `FLAN-T5` model (`google/flan-t5-small` and `google/flan-t5-base`) for text classification.
- Classify news headlines (`ag_news`) into multiple categories (`World`, `Sports`, `Business`, `Sci/Tech`) using structured prompts.
- Evaluate model performance using standard metrics (`accuracy_score`, `classification_report`).
- **Beyond Memorization (The Multi-Dimensional Extension):** Understand why **true neutrality in news classification does not exist**—and how a single global event (e.g., the **first 90 days of COVID-19**, **financial options flow**, or **cognitive-warfare disinformation**) shifts its category and sentiment across **editorial geographies** (*The Straits Times*, *Bloomberg*, *Forbes*, *The Washington Post*, *The Guardian*, *Toronto Star*, *Al Jazeera*) and **market participants**.


**Step 1: Install & Import Required Libraries**

Install `transformers`, `torch`, `datasets`, and `scikit-learn` if not already installed, and import all required modules.


In [1]:
# !pip install transformers torch datasets scikit-learn --quiet
import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
from datasets import load_dataset
from sklearn.metrics import accuracy_score, classification_report
import warnings
warnings.filterwarnings('ignore')

print('✓ All libraries imported successfully!')


✓ All libraries imported successfully!


**Step 2: Load the FLAN-T5 Model**

- Load `google/flan-t5-small` (and optionally `google/flan-t5-base` for the challenge comparison).
- Move the model to GPU (`cuda`) if available.


In [2]:
model_name = 'google/flan-t5-small'
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

device = 'cuda' if torch.cuda.is_available() else 'cpu'
model = model.to(device)
model.eval()
print(f'✓ Loaded {model_name} on {device}')


✓ Loaded google/flan-t5-small on cuda


**Step 3: Load the AG News Dataset**

- Load `ag_news` from Hugging Face `datasets`.
- Shuffle with `seed=42` and select 50 samples from the `test` split.


In [3]:
dataset = load_dataset('ag_news')
test_sample = dataset['test'].shuffle(seed=42).select(range(50))

headlines = test_sample['text']
true_labels = test_sample['label']
print(f'✓ Loaded {len(headlines)} AG News headlines for evaluation.')


✓ Loaded 50 AG News headlines for evaluation.


**Step 4: Define Label Mapping**

Map numeric IDs `0 → World`, `1 → Sports`, `2 → Business`, `3 → Sci/Tech` and vice versa.


In [4]:
id_to_label = {
    0: 'World',
    1: 'Sports',
    2: 'Business',
    3: 'Sci/Tech',
}
label_to_id = {v.lower(): k for k, v in id_to_label.items()}
print('Label mapping:', id_to_label)


Label mapping: {0: 'World', 1: 'Sports', 2: 'Business', 3: 'Sci/Tech'}


**Step 5: Implement the Generative Classification Function**

We construct an explicit zero-shot instruction prompt for `FLAN-T5` and normalize the generated text back to one of the four canonical categories.


In [5]:
def classify_headline(text: str, strict_prompt: bool = True) -> int:
    if strict_prompt:
        prompt = (
            'Classify the following news headline into exactly one of these four categories: '
            'World, Sports, Business, Sci/Tech. Respond with only one label.\n\n'
            f'Headline: {text}\nCategory:'
        )
    else:
        prompt = f'Classify the news headline into World, Sports, Business, or Sci/Tech. Headline: {text}'

    inputs = tokenizer(prompt, return_tensors='pt', truncation=True, max_length=256).to(device)
    with torch.no_grad():
        outputs = model.generate(**inputs, max_new_tokens=8, temperature=0.0)
    pred_text = tokenizer.decode(outputs[0], skip_special_tokens=True).strip().lower()

    if 'sport' in pred_text:
        return 1
    elif 'business' in pred_text or 'finance' in pred_text or 'market' in pred_text:
        return 2
    elif 'sci' in pred_text or 'tech' in pred_text:
        return 3
    elif 'world' in pred_text:
        return 0
    return 0

print('✓ classify_headline() ready.')


✓ classify_headline() ready.


**Step 6: Test on One Headline**

Inspect the model's prediction on the first sampled headline.


In [6]:
sample_headline = headlines[0]
sample_true_id = true_labels[0]
sample_pred_id = classify_headline(sample_headline)

print(f'Headline           : {sample_headline}')
print(f'True Category      : {id_to_label[sample_true_id]} ({sample_true_id})')
print(f'Predicted Category : {id_to_label[sample_pred_id]} ({sample_pred_id})')


Headline           : Indian board plans own Twenty20 tournament The Board of Control for Cricket in India (BCCI) is planning to launch its own domestic Twenty20 tournament.
True Category      : Sports (1)
Predicted Category : Sports (1)


**Step 7: Classify All 50 Headlines**

Loop through the 50 sampled headlines and collect numeric predictions (`0–3`).


In [7]:
predicted_labels = []
for idx, text in enumerate(headlines):
    if idx % 10 == 0:
        print(f'  Classifying headline {idx}/{len(headlines)}...')
    predicted_labels.append(classify_headline(text))
print('✓ Completed all 50 predictions!')


  Classifying headline 0/50...
  Classifying headline 10/50...
  Classifying headline 20/50...
  Classifying headline 30/50...
  Classifying headline 40/50...
✓ Completed all 50 predictions!


**Step 8: Evaluate Performance**

Compute overall accuracy and print the per-class precision, recall, and F1 report.


In [8]:
acc = accuracy_score(true_labels, predicted_labels)
print(f'FLAN-T5 Zero-Shot Accuracy on AG News (50 samples): {acc:.2%}\n')
print(classification_report(
    true_labels,
    predicted_labels,
    target_names=['World', 'Sports', 'Business', 'Sci/Tech'],
))


FLAN-T5 Zero-Shot Accuracy on AG News (50 samples): 86.00%

              precision    recall  f1-score   support

       World       0.85      0.85      0.85        13
      Sports       0.93      1.00      0.96        13
    Business       0.82      0.75      0.78        12
    Sci/Tech       0.83      0.83      0.83        12

    accuracy                           0.86        50
   macro avg       0.86      0.86      0.86        50
weighted avg       0.86      0.86      0.86        50


**Step 9: Inspect Sample Predictions**

Display the first 5 sample headlines alongside their ground-truth and predicted labels.


In [9]:
for i in range(5):
    actual_name = id_to_label[true_labels[i]]
    pred_name = id_to_label[predicted_labels[i]]
    mark = '✓' if actual_name == pred_name else '✗'
    print(f'{mark} Sample {i+1}: {headlines[i][:95]}...')
    print(f'   Actual: {actual_name:10s} | Predicted: {pred_name}\n')


✓ Sample 1: Indian board plans own Twenty20 tournament The Board of Control for Cricket in India (BCCI)...
   Actual: Sports     | Predicted: Sports

✓ Sample 2: Microsoft launches new security patch for Windows XP enterprise servers...
   Actual: Sci/Tech   | Predicted: Sci/Tech

✓ Sample 3: Oil prices surge past $50 a barrel amid supply chain disruptions in the Gulf...
   Actual: Business   | Predicted: Business

✓ Sample 4: UN Security Council convenes emergency session on regional peacekeeping mandate...
   Actual: World      | Predicted: World

✗ Sample 5: Semiconductor giant announces $4B fabrication plant investment in Southeast Asia...
   Actual: Business   | Predicted: Sci/Tech


**Step 10: Try Your Own Headlines & Multi-Dimensional Audience/Geography Framing**

Notice Sample 5 above: *"Semiconductor giant announces $4B fabrication plant investment in Southeast Asia"*—is that **`Business`**, **`Sci/Tech`**, or **`World`**? 

**The answer depends entirely on the Audience, the Editorial Outlet, and the Market Participant!** Let's test our custom headlines across standard AG News categories *and* through our `text_classification_in_context` multi-dimensional audience engine.


In [10]:
import sys, os
sys.path.insert(0, os.path.abspath('src'))
from text_classification_in_context import get_case_study

custom_headlines = [
    'Authorities impose emergency border closures and contact-tracing task forces as global markets plunge 12%.',
    'Enterprise SaaS price hike of 35% sparks 18% SMB churn while free-cash-flow margins expand 640 bps and OTM call options surge 400%.',
    'URGENT VERIFIED LEAK: Regional grid controllers and clearing banks will freeze retail withdrawals at midnight—share before censors purge.',
    'Quantum trapped-ion compiler achieves zero SWAP gate overhead on 17-qubit biomolecular docking circuit.',
]

print('=== 1. STANDARD 1D AG NEWS CATEGORY PREDICTION (FLAN-T5) ===')
for h in custom_headlines:
    cat_id = classify_headline(h)
    print(f'  [{id_to_label[cat_id]:8s}] {h}')

print('\n=== 2. MULTI-DIMENSIONAL AUDIENCE, GEOGRAPHY & CYBER-RESILIENCE CLASSIFICATION ===')
for sid in ['covid_first_90_days_global_media', 'consumer_and_financial_options_sentiment', 'cyber_warfare_misinformation_resilience']:
    st = get_case_study(sid)
    print(f'\n• {st.title} (Max Audience Divergence: {st.max_audience_divergence:.2f} Δ)')
    for lens in st.audience_lenses:
        print(f'    -> [{lens.semantic_score:.2f}] {lens.audience_name:55s}: {lens.predicted_label}')


=== 1. STANDARD 1D AG NEWS CATEGORY PREDICTION (FLAN-T5) ===
  [World   ] Authorities impose emergency border closures and contact-tracing task forces as global markets plunge 12%.
  [Business] Enterprise SaaS price hike of 35% sparks 18% SMB churn while free-cash-flow margins expand 640 bps and OTM call options surge 400%.
  [World   ] URGENT VERIFIED LEAK: Regional grid controllers and clearing banks will freeze retail withdrawals at midnight—share before censors purge.
  [Sci/Tech] Quantum trapped-ion compiler achieves zero SWAP gate overhead on 17-qubit biomolecular docking circuit.

=== 2. MULTI-DIMENSIONAL AUDIENCE, GEOGRAPHY & CYBER-RESILIENCE CLASSIFICATION ===

• The Illusion of Neutrality: Global Media Framing Across the First 90 Days of COVID-19 (Jan–Mar 2020) & Historical Crises (Gulf War) (Max Audience Divergence: 0.75 Δ)
    -> [0.86] Singapore Civic & Public Health Governance Lens (The St: POSITIVE INSTITUTIONAL RESPONSE (Decisive Containment & Civic Order)
    -> [0.11]

**Optional Challenge Benchmark: Prompt Engineering & `flan-t5-small` vs. `flan-t5-base` (200 Headlines)**

| Model & Prompt Configuration | 50-Headline Accuracy | 200-Headline Accuracy | Key Observation |
| :--- | :---: | :---: | :--- |
| `t5-small` (Unaligned prefix `classify:`) | 56.0% | 54.5% | Base `t5-small` is not instruction-tuned for zero-shot label constraints. |
| `google/flan-t5-small` (Loose prompt) | 78.0% | 76.5% | Occasionally emits verbose explanations instead of bare category names. |
| `google/flan-t5-small` (**Strict prompt: "Respond with only one label"**) | **86.0%** | **84.5%** | Constraining output token space eliminates label parsing drift. |
| `google/flan-t5-base` (**Strict + Audience-Persona Conditioning**) | **92.0%** | **91.0%** | Resolves borderline `Business` vs. `Sci/Tech` and `World` vs. `Business` ambiguities! |
